# Implement Sliding Window Attention — Solution

**Difficulty**: 🟡 Medium

**Companies**: Mistral, Anthropic, Google, DeepMind

---

### Problem Statement

Standard self-attention has **O(n²)** memory and compute complexity, which becomes prohibitive for long sequences. **Sliding window attention** (used in Mistral, Longformer, and others) restricts each token to only attend to a local window of **w** tokens on each side, reducing complexity to **O(n·w)**.

Your task is to implement sliding window attention by:
1. Creating a **band-diagonal attention mask** where token `i` can attend to tokens `[i-w, i+w]`.
2. Applying the mask to **scaled dot-product attention**.
3. Comparing the output with full attention to verify correctness.

---

### Requirements

1. **`create_sliding_window_mask(seq_len, window_size)`** — Returns a boolean mask of shape `(seq_len, seq_len)` where `mask[i][j] = True` if `|i - j| <= window_size`.
2. **`sliding_window_attention(q, k, v, window_size)`** — Applies scaled dot-product attention with the sliding window mask. Masked positions should get `-inf` before softmax.
3. **Memory comparison** — Show that sliding window uses O(n·w) non-zero entries vs O(n²) for full attention.

---

### Constraints

- ✅ Must handle batched inputs: `q, k, v` have shape `(batch_size, seq_len, d_model)`.
- ✅ When `window_size >= seq_len`, output must match full (unmasked) attention exactly.
- ✅ Attention weights for each query must sum to 1 (valid probability distribution).
- ❌ Do **not** use `torch.nn.MultiheadAttention` or any built-in attention layers.

---

<details>
  <summary>💡 Hint</summary>

  - Create the mask using: `mask = (torch.arange(seq_len).unsqueeze(0) - torch.arange(seq_len).unsqueeze(1)).abs() <= window_size`.
  - Apply the mask by setting `scores[~mask] = float('-inf')` before softmax.
  - Scale factor for attention: `1 / sqrt(d_model)`.
  - Non-zero entries in a band matrix of width 2w+1: approximately `n * (2w + 1)` (clamped at edges).

</details>

---

In [ ]:
import math
import torch
import torch.nn.functional as F

In [ ]:
# Setup
torch.manual_seed(42)

batch_size = 2
seq_len = 16
d_model = 32
window_size = 3  # attend to 3 tokens on each side

q = torch.randn(batch_size, seq_len, d_model)
k = torch.randn(batch_size, seq_len, d_model)
v = torch.randn(batch_size, seq_len, d_model)

print(f"Q/K/V shape: {q.shape}")
print(f"Window size: {window_size} (each token attends to {2 * window_size + 1} positions)")

In [ ]:
def create_sliding_window_mask(seq_len: int, window_size: int) -> torch.Tensor:
    """
    Create a sliding window attention mask.

    Args:
        seq_len:     sequence length
        window_size: number of tokens to attend to on each side

    Returns:
        Boolean tensor of shape (seq_len, seq_len) where True = allowed to attend.
    """
    positions = torch.arange(seq_len)
    # Compute pairwise absolute distances
    dist = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()
    # Mask: True where distance is within the window
    return dist <= window_size


def full_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Standard scaled dot-product attention (no mask).

    Args:
        q, k, v: (batch_size, seq_len, d_model)

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    d = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    weights = F.softmax(scores, dim=-1)
    return weights @ v


def sliding_window_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                              window_size: int) -> torch.Tensor:
    """
    Scaled dot-product attention with sliding window mask.

    Args:
        q, k, v:     (batch_size, seq_len, d_model)
        window_size: number of tokens to attend to on each side

    Returns:
        Output tensor of shape (batch_size, seq_len, d_model)
    """
    d = q.size(-1)
    seq_len = q.size(1)

    # Step 1: Compute raw attention scores
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)

    # Step 2: Create the sliding window mask
    mask = create_sliding_window_mask(seq_len, window_size)  # (seq_len, seq_len)

    # Step 3: Apply mask - set masked (False) positions to -inf
    scores = scores.masked_fill(~mask.unsqueeze(0), float('-inf'))

    # Step 4: Softmax and weighted sum
    weights = F.softmax(scores, dim=-1)
    return weights @ v


# Quick test
mask = create_sliding_window_mask(seq_len, window_size)
print(f"Mask shape: {mask.shape}")
print(f"Mask (first 8x8):\n{mask[:8, :8].int()}")

out = sliding_window_attention(q, k, v, window_size)
print(f"\nOutput shape: {out.shape}")

In [ ]:
# Validation

# Validation 1: When window_size >= seq_len, should match full attention
print("=== Full Window = Full Attention ===")
out_full = full_attention(q, k, v)
out_big_window = sliding_window_attention(q, k, v, window_size=seq_len)
assert torch.allclose(out_full, out_big_window, atol=1e-6), "Large window should match full attention!"
print("PASSED: window_size=seq_len matches full attention.\n")

# Validation 2: Attention weights sum to 1
print("=== Attention Weights Sum to 1 ===")
d = q.size(-1)
scores = q @ k.transpose(-2, -1) / math.sqrt(d)
mask = create_sliding_window_mask(seq_len, window_size)
scores = scores.masked_fill(~mask.unsqueeze(0), float('-inf'))
weights = F.softmax(scores, dim=-1)
weight_sums = weights.sum(dim=-1)
assert torch.allclose(weight_sums, torch.ones_like(weight_sums), atol=1e-6), "Weights must sum to 1!"
print("PASSED: All attention weight rows sum to 1.\n")

# Validation 3: Smaller window changes the output
print("=== Different Window Sizes Give Different Outputs ===")
out_w1 = sliding_window_attention(q, k, v, window_size=1)
out_w3 = sliding_window_attention(q, k, v, window_size=3)
assert not torch.allclose(out_w1, out_w3), "Different window sizes should give different outputs!"
print("PASSED: window_size=1 and window_size=3 produce different outputs.\n")

# Validation 4: Memory comparison
print("=== Memory Comparison ===")
for n in [128, 512, 2048, 8192]:
    w = 64
    full_entries = n * n
    m = create_sliding_window_mask(n, w)
    window_entries = m.sum().item()
    ratio = window_entries / full_entries * 100
    print(f"  seq_len={n:5d}, window={w}: {window_entries:>10,.0f} / {full_entries:>12,.0f} entries ({ratio:.1f}%)")
print("PASSED: Sliding window uses far fewer entries for long sequences.\n")

print("All tests passed!")